# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [3]:
df = pd.read_csv("data/AviationData.csv", encoding="latin1", low_memory=False)
df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [4]:
df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors="coerce")

cutoff_date = pd.Timestamp("1983-01-01")

df_clean = df[
    (df["Aircraft.Category"] == "Airplane") &
    (df["Amateur.Built"] == "No") &
    (df["Event.Date"] >= cutoff_date)
].copy()

df_clean = df_clean.reset_index(drop=True)

df_clean.shape

(21447, 31)

I filtered the dataset to match the client's scope. The analysis keeps only airplane records, removes amateur-built aircraft, and limits the data to events from 1983 onward. Since the dataset runs through 2023, this 1983 cutoff reflects the client's 40-year maximum aircraft lifetime assumption.

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [5]:
passenger_outcome_columns = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]


display(df_clean[passenger_outcome_columns].isna().sum())


df_clean[passenger_outcome_columns] = df_clean[passenger_outcome_columns].fillna(0)


df_clean["Total.Passengers"] = (
    df_clean["Total.Fatal.Injuries"]
    + df_clean["Total.Serious.Injuries"]
    + df_clean["Total.Minor.Injuries"]
    + df_clean["Total.Uninjured"]
)


df_clean["Fatal.Serious.Injuries"] = (
    df_clean["Total.Fatal.Injuries"]
    + df_clean["Total.Serious.Injuries"]
)


df_clean["Fatal.Serious.Injury.Fraction"] = np.nan


valid_passenger_rows = df_clean["Total.Passengers"] > 0


df_clean.loc[
    valid_passenger_rows,
    "Fatal.Serious.Injury.Fraction",
] = (
    df_clean.loc[valid_passenger_rows, "Fatal.Serious.Injuries"]
    / df_clean.loc[valid_passenger_rows, "Total.Passengers"]
)


df_clean[
    [
        "Total.Fatal.Injuries",
        "Total.Serious.Injuries",
        "Total.Minor.Injuries",
        "Total.Uninjured",
        "Total.Passengers",
        "Fatal.Serious.Injuries",
        "Fatal.Serious.Injury.Fraction",
    ]
].head()


Total.Fatal.Injuries      2750
Total.Serious.Injuries    2828
Total.Minor.Injuries      2544
Total.Uninjured            711
dtype: int64

,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Total.Passengers,Fatal.Serious.Injuries,Fatal.Serious.Injury.Fraction
0,0.0,0.0,0.0,588.0,588.0,0.0,0.0
1,0.0,0.0,0.0,588.0,588.0,0.0,0.0
2,1.0,1.0,0.0,0.0,2.0,2.0,1.0
3,1.0,0.0,0.0,4.0,5.0,1.0,0.2
4,0.0,0.0,0.0,289.0,289.0,0.0,0.0


I created a fatal or serious injury metric to measure accident severity relative to the estimated number of people involved. To estimate the total number of people involved in each accident, I added the recorded fatal injuries, serious injuries, minor injuries, and uninjured counts.

Missing values in these injury columns were filled with 0 so they would not add to the total passenger estimate. I then calculated the fatal or serious injury fraction by dividing fatal plus serious injuries by the estimated total passengers. Rows with 0 estimated passengers were left as missing for this fraction to avoid dividing by zero. This assumes that missing injury counts represent no recorded injuries of that type rather than unavailable data, so the fraction may be slightly biased for rows with incomplete injury reporting.

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [6]:
df_clean["Aircraft.damage"].value_counts(dropna=False)

Aircraft.damage
Substantial    16990
Destroyed       2316
NaN             1227
Minor            817
Unknown           97
Name: count, dtype: int64

In [7]:
df_clean["Aircraft.damage"] = df_clean["Aircraft.damage"].fillna("Unknown")

df_clean["Aircraft.Destroyed"] = (
    df_clean["Aircraft.damage"]
    .str.strip()
    .eq("Destroyed")
)

df_clean[
    [
        "Aircraft.damage",
        "Aircraft.Destroyed",
    ]
].head()

,Aircraft.damage,Aircraft.Destroyed
0,Minor,False
1,Minor,False
2,Destroyed,True
3,Unknown,False
4,Minor,False


I created an Aircraft.Destroyed column to track whether each accident resulted in the aircraft being destroyed. Missing aircraft damage values were filled with Unknown so those records would remain in the cleaned dataset instead of being dropped.

Records labeled Destroyed were marked as True, while all other damage outcomes were marked as False. This treats unknown damage outcomes as not confirmed destroyed, which keeps the metric conservative while still allowing those rows to remain available for later analysis.

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [9]:
df_clean["Make"] = df_clean["Make"].str.strip().str.upper()

make_name_replacements = {
    "AIR TRACTOR INC": "AIR TRACTOR",
    "AIR TRACTOR INC.": "AIR TRACTOR",
    "AIR TRACTOR, INC.": "AIR TRACTOR",
    "CIRRUS DESIGN CORP": "CIRRUS",
    "CIRRUS DESIGN CORP.": "CIRRUS",
    "DEHAVILLAND": "DE HAVILLAND",
    "AVIAT AIRCRAFT INC": "AVIAT",
    "BOMBARDIER INC": "BOMBARDIER",
    "MOONEY AIRCRAFT CORP": "MOONEY",
    "MOONEY AIRCRAFT CORP.": "MOONEY",
    "MOONEY AIRCRAFT CORPORATION": "MOONEY",
}

df_clean["Make"] = df_clean["Make"].replace(make_name_replacements)

make_counts = df_clean["Make"].value_counts()

common_makes = make_counts[make_counts >= 50].index

df_clean = df_clean[df_clean["Make"].isin(common_makes)].copy()

df_clean = df_clean.reset_index(drop=True)

df_clean["Make"].value_counts().head(50)

Make
CESSNA                            7146
PIPER                             3989
BEECH                             1431
BOEING                            1264
AIR TRACTOR                        432
MOONEY                             399
CIRRUS                             385
AIRBUS                             243
BELLANCA                           219
MAULE                              215
AERONCA                            200
DE HAVILLAND                       168
CHAMPION                           158
EMBRAER                            153
GRUMMAN                            147
AVIAT                              146
LUSCOMBE                           141
STINSON                            129
BOMBARDIER                         115
MCDONNELL DOUGLAS                  108
NORTH AMERICAN                     106
TAYLORCRAFT                         93
AERO COMMANDER                      90
SOCATA                              75
DIAMOND AIRCRAFT IND INC            74
RAYTHEON AIRCRAFT CO

I inspected the Make column to identify missing values, inconsistent formatting, and manufacturer names with low record counts. Several manufacturers appeared multiple ways because of capitalization, punctuation, spacing, or company suffixes.

To clean this column, I removed extra spacing, converted manufacturer names to uppercase, and combined clear manufacturer name variants. I did not automatically merge every similar name because fuzzy matches can incorrectly combine different companies. For the analysis, I kept only makes with at least 50 accident records. This threshold keeps the analysis focused on manufacturers with enough records to support more stable comparisons, while removing rare makes that could create misleading rates from only a small number of accidents.

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized